# PHS 312 — toolkit self-test

One cell. Press the play button, wait about a minute, then copy everything
it prints and paste it back.

This checks that every RDKit and py3Dmol call the rebuilt notebooks will use
actually works in Colab, before any of them are built on it.


In [ ]:
#@title Run this one cell, then copy everything it prints  { display-mode: "form" }
#@markdown Installs RDKit and py3Dmol, then exercises every call the rebuilt
#@markdown PHS 312 notebooks will use. Takes about a minute.
import subprocess, sys, time, traceback
t0=time.time()
subprocess.run([sys.executable,"-m","pip","install","-q","rdkit","py3Dmol"],check=False)
print(f"install: {time.time()-t0:.0f}s\n")

ok=[]; bad=[]
def check(label, fn):
    try:
        r=fn(); ok.append(label); print(f"[PASS] {label}: {r}")
    except Exception as e:
        bad.append((label,repr(e))); print(f"[FAIL] {label}: {type(e).__name__}: {e}")

import numpy as np
import rdkit, py3Dmol
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
RDLogger.DisableLog("rdApp.*")
print("rdkit", rdkit.__version__, "| py3Dmol", getattr(py3Dmol,"__version__","?"),
      "| numpy", np.__version__, "\n")

TAC="NC1=C2CCCCC2=NC3=CC=CC=C13"

def t_embed():
    m=Chem.AddHs(Chem.MolFromSmiles(TAC))
    p=AllChem.ETKDGv3(); p.randomSeed=0xf00d
    rc=AllChem.EmbedMolecule(m,p)
    return f"rc={rc} atoms={m.GetNumAtoms()} heavy={m.GetNumHeavyAtoms()}"

def t_mmff():
    m=Chem.AddHs(Chem.MolFromSmiles(TAC))
    p=AllChem.ETKDGv3(); p.randomSeed=0xf00d; AllChem.EmbedMolecule(m,p)
    pr=AllChem.MMFFGetMoleculeProperties(m)
    ff=AllChem.MMFFGetMoleculeForceField(m,pr)
    e0=ff.CalcEnergy(); its=ff.Minimize(maxIts=5000); e1=ff.CalcEnergy()
    return f"E {e0:.1f} -> {e1:.1f} kcal/mol (released {e0-e1:.1f}), minimize rc={its}"

def t_multiconf():
    m=Chem.AddHs(Chem.MolFromSmiles("CCCCCCO"))
    p=AllChem.ETKDGv3(); p.randomSeed=1
    ids=AllChem.EmbedMultipleConfs(m,numConfs=20,params=p)
    res=AllChem.MMFFOptimizeMoleculeConfs(m,maxIters=800)
    es=[e for conv,e in res]
    return f"{len(ids)} confs, E range {min(es):.1f} to {max(es):.1f}"

def t_rms():
    m=Chem.AddHs(Chem.MolFromSmiles("CCCCCCO"))
    p=AllChem.ETKDGv3(); p.randomSeed=1
    AllChem.EmbedMultipleConfs(m,numConfs=5,params=p)
    r=AllChem.GetConformerRMS(m,0,1,prealigned=False)
    return f"RMSD conf0 vs conf1 = {r:.2f} A"

def t_gasteiger():
    m=Chem.AddHs(Chem.MolFromSmiles(TAC))
    AllChem.ComputeGasteigerCharges(m)
    q=[a.GetDoubleProp("_GasteigerCharge") for a in m.GetAtoms()]
    q=np.nan_to_num(np.array(q))
    return f"sum={q.sum():+.3f} min={q.min():+.3f} max={q.max():+.3f}"

def t_draw():
    m=Chem.MolFromSmiles(TAC); AllChem.Compute2DCoords(m)
    d=rdMolDraw2D.MolDraw2DSVG(300,200)
    rdMolDraw2D.PrepareAndDrawMolecule(d,m,highlightAtoms=[0,1],
        highlightAtomColors={0:(1.,.6,.6),1:(.6,.6,1.)})
    d.FinishDrawing()
    return f"SVG {len(d.GetDrawingText())} chars"

def t_desc():
    m=Chem.MolFromSmiles(TAC)
    return (f"MolWt={Descriptors.MolWt(m):.1f} logP={Descriptors.MolLogP(m):.2f} "
            f"TPSA={Descriptors.TPSA(m):.1f} rotB={rdMolDescriptors.CalcNumRotatableBonds(m)}")

def t_pdb():
    import urllib.request
    with urllib.request.urlopen("https://files.rcsb.org/download/1ACJ.pdb",timeout=60) as r:
        txt=r.read().decode("utf-8","replace")
    lig=[l for l in txt.splitlines() if l.startswith("HETATM") and l[17:20].strip()=="THA"]
    return f"1ACJ fetched, {len(txt.splitlines())} lines, THA ligand {len(lig)} atoms"

def t_py3dmol():
    import urllib.request
    with urllib.request.urlopen("https://files.rcsb.org/download/1ACJ.pdb",timeout=60) as r:
        txt=r.read().decode("utf-8","replace")
    v=py3Dmol.view(width=420,height=300)
    v.addModel(txt,"pdb")
    v.setStyle({"cartoon":{"color":"spectrum"}})
    v.addStyle({"resn":"THA"},{"stick":{"radius":0.2}})
    v.addSurface(py3Dmol.VDW,{"opacity":0.6},{"resn":"THA"})
    v.zoomTo({"resn":"THA"})
    v.show()
    return "viewer rendered (you should see a 3D protein above)"

for lab,fn in [("embed 3D",t_embed),("MMFF94 minimise",t_mmff),
               ("multi-conformer",t_multiconf),("conformer RMSD",t_rms),
               ("Gasteiger charges",t_gasteiger),("2D drawing + highlight",t_draw),
               ("descriptors",t_desc),("fetch PDB 1ACJ",t_pdb),
               ("py3Dmol 3D viewer",t_py3dmol)]:
    check(lab,fn)

print("\n" + "="*58)
print(f"PASSED {len(ok)}/{len(ok)+len(bad)}")
if bad:
    print("FAILURES:")
    for l,e in bad: print("  -",l,"->",e)
else:
    print("All calls work. Paste this whole output back to Claude.")
print("="*58)